# G1: Inside the game engine

How the Hoopformer game turns 1,230 real games into simulated ones, in six steps. Most cells are a tour: run them and read the output. The **✍️ YOUR CODE** cells are exercises, and the **✅ CHECK** cell after each one tells you if you got it.

Needs the full 2025-26 season downloaded (the Studio has it). Use the **Hoopformer (.venv)** kernel.

In [ ]:
# Setup: run this first. It moves to the project folder so paths like data/... work,
# and reloads project code automatically whenever src/hoopformer/ changes.
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print("working in", Path.cwd())

## 1. Counting a real game

The engine learns from **chances**: a possession's first try, plus one more after every offensive rebound. `count_game` walks a game's play-by-play and counts what every player did, and on how many chances they were on the floor.

In [ ]:
import json
from hoopformer.game.actions import count_game, ZONES, EVENTS

OKC, HOU, SGA = 1610612760, 1610612745, 1628983
pbp = json.load(open("data/raw/playbyplayv3/0022500001.json"))
box = json.load(open("data/raw/boxscoretraditionalv3/0022500001.json"))["boxScoreTraditional"]
counts = count_game(pbp, box)
dict(counts.players[SGA])   # Shai Gilgeous-Alexander's counts in the 2025-26 opener

### ✍️ Exercise 1: rebuild OKC's score from the counts
Points = 2 × (rim makes + midrange makes) + 3 × three makes + free-throw makes. Add that up over every OKC player in `box["homeTeam"]["players"]` (OKC was at home) and store it in `okc_points`.

Hint: `counts.players.get(person_id, {})` gives a player's counts, and `.get("rim_made", 0)` reads one of them.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 1
print("rebuilt:", okc_points, "| official:", box["homeTeam"]["statistics"]["points"])
assert okc_points == box["homeTeam"]["statistics"]["points"], "every point should be in the counts"

## 2. Shrinkage: don't trust small samples

A player who goes 10-for-20 on threes is probably not a 50% shooter. The model mixes each player's record with `k` league-average attempts:

    rate = (makes + k × league rate) / (attempts + k)

For threes, `k = 150` (`PRIOR["three"]` in `src/hoopformer/game/model.py`).

### ✍️ Exercise 2
Write `shrunk_three(makes, attempts, league_rate)` using that formula with k = 150, then compute the shrunk rate for 10/20 and for 400/1000, with a league rate of 0.36. Store them in `small` and `large`.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 2
from hoopformer.game.model import shrink, PRIOR
print(f"10/20 -> {small:.3f} (raw 0.500) | 400/1000 -> {large:.3f} (raw 0.400)")
assert abs(small - shrink(10, 20, 0.36, PRIOR["three"])) < 1e-9 and abs(large - shrink(400, 1000, 0.36, PRIOR["three"])) < 1e-9
assert small < 0.40 < large + 0.01, "20 shots barely move the estimate; 1,000 shots mostly speak for themselves"

## 3. Fitting the model on the whole season

`season_counts` counts every game of the season (about 10 seconds), and `fit_action_model` turns the counts into rates.

In [ ]:
from hoopformer.game.actions import season_counts
from hoopformer.game.model import fit_action_model

players, teams, extras = season_counts(Path("data"), "2025-26")
model = fit_action_model(players, teams, extras, "2025-26")
league = model.league
print({z: round(league.make[z], 3) for z in ZONES}, "| FT", round(league.ft_pct, 3), "| home win", round(league.home_win_share, 3))
players.head()   # raw counts: one row per player, one column per thing counted

### ✍️ Exercise 3: the best finisher at the rim
Among players with **at least 200 rim attempts** (the `rim` column of `players`), find the one with the highest raw rim make rate (`rim_made / rim`). Store their person id in `best_rim_id`, then print `model.players[best_rim_id].name`.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 3
eligible = players[players["rim"] >= 200]
expected = (eligible["rim_made"] / eligible["rim"]).idxmax()
print(model.players[best_rim_id].name, f"{players.loc[best_rim_id, 'rim_made'] / players.loc[best_rim_id, 'rim']:.3f}")
assert best_rim_id == expected

## 4. Playing a game

One loop, one random generator, one seed: **the same seed replays the same game.**

In [ ]:
from hoopformer.game.engine import Game

result = Game(model, OKC, HOU, seed=7).play()
print(f"{result.away.tricode} {result.away.points} @ {result.home.tricode} {result.home.points}")
for event in result.events[:12]:
    minutes, seconds = divmod(int(event.clock), 60)
    print(f"Q{event.period} {minutes:2d}:{seconds:02d} {event.team} {event.text}")
result.box_score(result.home).head(8)

## 5. Inside one chance

On every chance, each of the five offensive players has a weight for each event (`rim`, `mid`, `three`, `turnover`, `free_throws`), and so does a team turnover. The engine picks one in proportion to the weights. The defense scales turnovers and free throws by its own factors.

### ✍️ Exercise 4: how often does SGA act?
For OKC's starting five (`starters`, below), compute the share of all weights that belong to SGA when OKC attacks Houston. Store it in `p_sga`.

Recipe, mirroring `Game._chance` in `src/hoopformer/game/engine.py`:
1. For each player `pid` in `starters`, for each event:
   - start from `w = model.players[pid].events[event]`
   - multiply turnovers by `hou_defense.turnover_factor`
   - multiply free throws by `hou_defense.free_throw_factor`
2. Add one team-turnover weight: `league.team_turnover_rate * hou_defense.turnover_factor`.
3. `p_sga` = SGA's weights ÷ all weights.

Use `starters` and `hou_defense` from the cell below.

In [ ]:
from hoopformer.game.engine import default_roster, minute_shares
shares = minute_shares([model.players[p] for p in default_roster(model, OKC)])
starters = sorted(shares, key=shares.get, reverse=True)[:5]
hou_defense = model.teams[HOU]
print("OKC starters:", [model.players[p].name for p in starters])

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 4
weights = {}
for pid in starters:
    for event in EVENTS:
        w = model.players[pid].events[event]
        w *= hou_defense.turnover_factor if event == "turnover" else hou_defense.free_throw_factor if event == "free_throws" else 1
        weights[(pid, event)] = w
total = sum(weights.values()) + league.team_turnover_rate * hou_defense.turnover_factor
expected = sum(w for (pid, _), w in weights.items() if pid == SGA) / total
print(f"SGA acts on {p_sga:.1%} of OKC's chances with this lineup (five players would average 20%)")
assert abs(p_sga - expected) < 1e-9

## 6. Your first experiment: a "coach lever"

Coaching instructions will become changes to these rates. Here's a preview: make SGA a better midrange shooter and measure what it's worth in wins.

**The trick: use the same seeds for both versions.** Each game then starts from the same dice rolls, so the comparison measures the change, not luck. This is called common random numbers.

### ✍️ Exercise 5
1. Simulate 1,000 OKC-vs-HOU games with seeds 0 to 999 and store OKC's win share in `win_before`.
2. Make a copy: `better = copy.deepcopy(model)`, then raise `better.players[SGA].make["mid"]` by 0.05.
3. Simulate the same 1,000 seeds with `better` and store the win share in `win_after`.

Each game takes about 2 ms.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 5
print(f"OKC wins {win_before:.1%} -> {win_after:.1%} when SGA's midrange make rate rises 5 points")
assert 0 < win_before < 1 and win_after > win_before, "with the same seeds, a better shooter should win more"

## 7. Is the simulated league realistic?

The test the engine has to pass: a simulated league must match the real season on 16 statistics.

In [ ]:
from hoopformer.game.realism import compare, real_per_team_game, simulate_league, simulated_per_team_game
table = compare(real_per_team_game(extras), simulated_per_team_game(simulate_league(model, 500, seed=1)))
table[["statistic", "real", "simulated", "difference", "tolerance", "ok"]].round(4)

**Done.** Things to think about for day 3 (coach language):
- Which rates would "attack the rim more" change?
- Which would "switch everything on defense" change?
- Exercise 5 showed how to measure any lever's worth: the same seeds, before and after.